In [1]:
from torch import nn
import torch

# 块

## 顺序块
### 官方Sequential

In [2]:
net1 = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))

l = [nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10)]
net2 = nn.Sequential(*l)

net1, net2

(Sequential(
   (0): Linear(in_features=20, out_features=256, bias=True)
   (1): ReLU()
   (2): Linear(in_features=256, out_features=10, bias=True)
 ),
 Sequential(
   (0): Linear(in_features=20, out_features=256, bias=True)
   (1): ReLU()
   (2): Linear(in_features=256, out_features=10, bias=True)
 ))

In [15]:
# 分开添加
net = nn.Sequential()
net.add_module("myblock0", nn.Linear(20, 256))
net.add_module("myblock1", nn.ReLU())
net.add_module("myblock2", nn.Linear(256, 10))
net

Sequential(
  (myblock0): Linear(in_features=20, out_features=256, bias=True)
  (myblock1): ReLU()
  (myblock2): Linear(in_features=256, out_features=10, bias=True)
)

可以指定输出 module 部分

In [16]:
net.myblock0

Linear(in_features=20, out_features=256, bias=True)

### 自定义Sequential

In [5]:
class MySequential(nn.Module):
    def __init__(self, *args):
        super().__init__()
        for idx, module in enumerate(args):
            # 这里，module是Module子类的一个实例。我们把它保存在'Module'类的成员
            # 变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self, X):
        # OrderedDict保证了按照成员添加的顺序遍历它们
        for block in self._modules.values():
            X = block(X)
        return X

net = MySequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
X = torch.rand(2, 20)
net(X)

tensor([[-0.1012, -0.0873,  0.1588,  0.0115, -0.0969, -0.1049,  0.0654, -0.1116,
          0.0581, -0.0134],
        [-0.1430, -0.0433,  0.1650,  0.1072,  0.0351, -0.1098,  0.1765, -0.1762,
         -0.0725,  0.0706]], grad_fn=<AddmmBackward0>)

## 自定义块

In [3]:
class MLP(nn.Module):
    # 用模型参数声明层。这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Module的构造函数来执行必要的初始化。
        # 这样，在类实例化时也可以指定其他函数参数，例如模型参数params（稍后将介绍）
        super().__init__()
        self.hidden = nn.Linear(20, 256)  # 隐藏层
        self.relu = nn.ReLU()
        self.out = nn.Linear(256, 10)  # 输出层

    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self, X):
        return self.out(self.relu(self.hidden(X)))


net = MLP()
X = torch.rand(2, 20)
net(X)

tensor([[-0.0279, -0.0293, -0.0024,  0.1012,  0.1230, -0.0053,  0.1371, -0.0447,
         -0.1586,  0.1932],
        [-0.0045,  0.0374,  0.0239,  0.2165,  0.0657,  0.0293,  0.1624, -0.0449,
         -0.1148,  0.1416]], grad_fn=<AddmmBackward0>)

换一种ReLU的写法

In [8]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(20, 256)
        self.out = nn.Linear(256, 10)

    def forward(self, X):
        # ReLU要写成这样
        return self.out(nn.ReLU()(self.hidden(X)))


net = MLP()
X = torch.rand(2, 20)
net(X)

tensor([[ 0.2875, -0.1056,  0.0569, -0.1080, -0.1476,  0.0502, -0.1791, -0.3067,
         -0.1276,  0.1145],
        [ 0.3368, -0.1305, -0.0451, -0.0857, -0.2732, -0.0616, -0.1437, -0.1544,
         -0.3183,  0.1924]], grad_fn=<AddmmBackward0>)

特殊梯度计算

In [27]:
class FixedHiddenMLP(nn.Module):
    def __init__(self):
        super().__init__()
        # 不计算梯度的随机权重参数。因此其在训练期间保持不变
        self.rand_weight = torch.rand((20, 20), requires_grad=False)
        self.linear = nn.Linear(20, 20)

    def forward(self, X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = torch.matmul(X, self.rand_weight) + 1
        # 复用全连接层。这相当于两个全连接层共享参数
        X = self.linear(X)
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

net = FixedHiddenMLP()
X = torch.rand(2, 20)
net(X)

tensor(0.0992, grad_fn=<SumBackward0>)

权重参数网络

In [ ]:
class SynthesizedImage(nn.Module):
    def __init__(self, img_shape, **kwargs):
        super(SynthesizedImage, self).__init__(**kwargs)
        self.weight = nn.Parameter(torch.rand(*img_shape))

    # forward 不需要 X， 只是传递权重
    def forward(self):
        return self.weight

shape = (300,400)
net = SynthesizedImage(shape)
net()

Parameter containing:
tensor([[0.7720, 0.7352, 0.5330,  ..., 0.5928, 0.2361, 0.1135],
        [0.8262, 0.9850, 0.7148,  ..., 0.1940, 0.4422, 0.0151],
        [0.0601, 0.7996, 0.2169,  ..., 0.5041, 0.8360, 0.5286],
        ...,
        [0.5656, 0.1060, 0.4269,  ..., 0.5850, 0.5531, 0.2456],
        [0.8157, 0.7269, 0.3399,  ..., 0.1308, 0.9227, 0.6943],
        [0.8362, 0.6395, 0.2451,  ..., 0.5258, 0.9058, 0.7010]],
       requires_grad=True)

## 嵌套

In [19]:
def block1():
    return nn.Sequential(nn.Linear(4, 8), nn.ReLU(),
                         nn.Linear(8, 4), nn.ReLU())


def block2():
    net = nn.Sequential(block1(), nn.Linear(4, 4), block1())
    return net


net = nn.Sequential(block2(), nn.Linear(4, 1))
net

Sequential(
  (0): Sequential(
    (0): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
    (1): Linear(in_features=4, out_features=4, bias=True)
    (2): Sequential(
      (0): Linear(in_features=4, out_features=8, bias=True)
      (1): ReLU()
      (2): Linear(in_features=8, out_features=4, bias=True)
      (3): ReLU()
    )
  )
  (1): Linear(in_features=4, out_features=1, bias=True)
)

In [25]:
net = nn.Sequential()
net.add_module('features', block2())
net.add_module('output', nn.Linear(4, 1))
net, net.features, net.output

(Sequential(
   (features): Sequential(
     (0): Sequential(
       (0): Linear(in_features=4, out_features=8, bias=True)
       (1): ReLU()
       (2): Linear(in_features=8, out_features=4, bias=True)
       (3): ReLU()
     )
     (1): Linear(in_features=4, out_features=4, bias=True)
     (2): Sequential(
       (0): Linear(in_features=4, out_features=8, bias=True)
       (1): ReLU()
       (2): Linear(in_features=8, out_features=4, bias=True)
       (3): ReLU()
     )
   )
   (output): Linear(in_features=4, out_features=1, bias=True)
 ),
 Sequential(
   (0): Sequential(
     (0): Linear(in_features=4, out_features=8, bias=True)
     (1): ReLU()
     (2): Linear(in_features=8, out_features=4, bias=True)
     (3): ReLU()
   )
   (1): Linear(in_features=4, out_features=4, bias=True)
   (2): Sequential(
     (0): Linear(in_features=4, out_features=8, bias=True)
     (1): ReLU()
     (2): Linear(in_features=8, out_features=4, bias=True)
     (3): ReLU()
   )
 ),
 Linear(in_features=4, o

## nn

In [ ]:
nn.ReLU()

nn.Flatten()

nn.Dropout(0.1)

nn.Linear(256, 10)

nn.Conv2d(1, 1, kernel_size=3)

nn.MaxPool2d(2)

nn.AvgPool2d(2)

# 100通道数
nn.BatchNorm1d(100)

## 打印


In [3]:
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
# 整体
print(net)

# 某层
print(net[2])

Sequential(
  (0): Linear(in_features=20, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=10, bias=True)
)
Linear(in_features=256, out_features=10, bias=True)


In [7]:
# 1个随机样本
X = torch.rand(size=(1, 20), dtype=torch.float32)
for layer in net:
    X = layer(X)
    print(f'output shape: {layer.__class__.__name__: <15}{X.shape}')

output shape: Linear         torch.Size([1, 256])
output shape: ReLU           torch.Size([1, 256])
output shape: Linear         torch.Size([1, 10])
